## Trabajo Práctico Nº 5

### CASTRO NICOLAS, MENDUIÑA JUAN, SCHIAVONI JUAN LUIS

IMPORTAMOS DIVERSOS PAQUETES QUE SERÁN UTILIZADOS A LO LARGO DEL TP

In [ ]:
import os
import re
import string
import pandas as pd
import numpy as np
import tweepy

import sys
#!{sys.executable} -m pip install wordcloud
from wordcloud import WordCloud
#!{sys.executable} -m pip install Pillow
from PIL import Image
import matplotlib.pyplot as plt

from textblob import TextBlob
from datetime import datetime
from datetime import timezone as tz
#!{sys.executable} -m pip install sentiment_analysis_spanish
#from sentiment_analysis_spanish import sentiment_analysis

import nltk
#nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

DEFINIMOS ALGUNAS FUNCIONES QUE SERÁN UTILIZADAS A LO LARGO DEL TP

In [ ]:
# FUNCIÓN "GET_ALL_TWEETS"

def get_all_tweets(screen_name, start_date):

    '''
    Esta función recibe el nombre de la persona de quien queremos extraer los tweets y devuelve una lista con todos los tweets y sus datos.
    Input:
      screen_name (str): nombre de la persona en twitter.
    Output:
      all_tweets (lista): lista con todos los tweets extraídos.
    '''

    # Solicitamos los 200 tweets más recientes (200 es el máximo permitido en count)
    new_tweets = api.user_timeline(screen_name=screen_name, tweet_mode="extended", count=200)

    # Creamos una lista para almacenar TODOS los tweets y agrego los recién extraidos
    all_tweets = []
    all_tweets.extend(new_tweets)

    # Guardamos el id del último tweet extraído 
    oldest = all_tweets[-1].id 

    # Extraemos tweets de a 200 hasta que no haya más
    while len(new_tweets) > 0 and all_tweets[-1].created_at > start_date:
        # Solicitamos 200 tweets más y los agrego a la lista de "all_tweets"
        new_tweets = api.user_timeline(screen_name=screen_name, tweet_mode="extended", count=200, max_id=oldest-1)
        all_tweets.extend(new_tweets)
        # Actualizamos el id del último tweet extraído
        oldest = all_tweets[-1].id 
        print("Hasta ahora se han extraído %s tweets" % len(all_tweets))

    return all_tweets

In [ ]:
# FUNCIÓN "SAVE_TWEETS_TEXT"

def save_tweets_text(all_tweets, csv_file=None):

    '''
    Esta función guarda los tweets en un data frame y, si se especifica un archivo .csv, también se guardarán ahí. 
    Input:
        all_tweets (lista): lista con tweets y sus datos.
        csv_file ('str'): nombre del archivo .csv.
    Output:
        df_all_tweets (df): tweets ordenados en una tabla con datos seleccinados.
    '''

    all_tweets_selection = []

    for tweet in all_tweets:
        one_tweet = [tweet.id_str, tweet.created_at, tweet.full_text, tweet.retweeted, tweet.favorite_count, tweet.in_reply_to_screen_name]
        all_tweets_selection.append(one_tweet)

    df_all_tweets = pd.DataFrame(all_tweets_selection)
    df_all_tweets.columns = ["id_str", "created_at", "text", "retweeted", "favorite_count", "in_reply_to_screen_name"]
    if csv_file:
        df_all_tweets.to_csv(csv_file, index=False, encoding="utf-8")

    return df_all_tweets

In [ ]:
# FUNCIÓN "CLEAN_TWEET"

def clean_tweet(tweet):

    '''
    Esta función limpia el texto de un tweet. Elimina caracteres especÍficos que
    se utilizan en twitter como los de re-tweets, los links y otros Non-ASCII. Devuelve el texto "limpio".
    Input:
        tweet (str): Texto del tweet original.
    Output:
        tweet (str): Texto del tweet limpiado.
    '''

    # Eliminamos los links
    tweet = re.sub(r"https\S+", "", tweet)
    # Eliminamos caracteres de re-tweets
    tweet = re.sub(r"^RT .*:", "", tweet)
    tweet = re.sub(r"@\S+", "", tweet)
    tweet = re.sub(r":", "", tweet)
    tweet = re.sub(r"‚Ä¶", "", tweet)
    # Reemplazamos caracteres non-ASCII con espacio
    tweet = re.sub(r"[^\x00-\x7F]+"," ", tweet)

    return tweet

ABRIMOS ARCHIVO .txt CON LAS KEYS DE TWITTER

In [ ]:
# Creamos variables que contienen mis claves de autenticación con la API
with open("twitter_keys.txt") as tw_k:
    consumer_key = tw_k.readline().strip()
    consumer_secret = tw_k.readline().strip()
    access_key = tw_k.readline().strip()
    access_secret = tw_k.readline().strip()

# Pasamos nuestras credenciales de twitter a tweepy
auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
auth.set_access_token(access_key, access_secret)
api = tweepy.API(auth)

DECLARAMOS EMOTICONES Y EMOJIS

In [ ]:
# Emoticones Contentos
emoticons_happy = set([
    ':-)', ':)', ';)', ':o)', ':]', ':3', ':c)', ':>', '=]', '8)', '=)', ':}',
    ':^)', ':-D', ':D', '8-D', '8D', 'x-D', 'xD', 'X-D', 'XD', '=-D', '=D',
    '=-3', '=3', ':-))', ":'-)", ":')", ':*', ':^*', '>:P', ':-P', ':P', 'X-P',
    'x-p', 'xp', 'XP', ':-p', ':p', '=p', ':-b', ':b', '>:)', '>;)', '>:-)',
    '<3'
    ])

# Emoticones Tristes
emoticons_sad = set([
    ':L', ':-/', '>:/', ':S', '>:[', ':@', ':-(', ':[', ':-||', '=L', ':<',
    ':-[', ':-<', '=\\', '=/', '>:(', ':(', '>.<', ":'-(", ":'(", ':\\', ':-c',
    ':c', ':{', '>:\\', ';('
    ])

# Combinamos emoticones contentos y tristes
emoticons = emoticons_happy.union(emoticons_sad)

# Emoji Patterns
emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # símbolos & pictogramas
                           u"\U0001F680-\U0001F6FF"  # transporte & símbolos mapas
                           u"\U0001F1E0-\U0001F1FF"  # banderas (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)

emoji_pattern_happy = re.compile("["
                           u"\U0001F600-\U0001F60F"  # emoticons
                           u"\U0001F617-\U0001F61D"  # emoticons
                           "]+", flags=re.UNICODE)
emoji_pattern_sad = re.compile("["
                           u"\U0001F610-\U0001F616"  # emoticons
                           u"\U0001F620-\U0001F62F"  # emoticons
                           "]+", flags=re.UNICODE)

#### Ejercicio 1:
Creen una función que limpie cualquier tipo de texto. Esta debe eliminar emoticones, emojis, palabras vacías (también llamadas stop words en la libreria de nltk), puntuaciones, etc. 

Además, cuando el parametro is_tweet este seteado en True, la función debe eliminar indicaciones de retweets, links y cualquier otra particularidad de los tweets que crean relevante eliminar. 

Prueben el funcionamiento de su función con el texto a continuación

In [ ]:
# Realizamos la prueba con el texto solicitado, y con otro alternativo que refiere a un tweet
texto = "She's got a smile :) that it seems to me. Reminds me of childhood memories. Where everything was as fresh as the bright blue sky. Now and then when I see her face \U0001F600 . She takes me away to that special place. And if I stare too long, I'd probably break down and cry :'( Whoa, oh, oh. Sweet child o' mine. Whoa, oh, oh, oh. Sweet love of mine. She's got eyes of the bluest skies. As if they thought of rain. I'd hate to look into those eyes and see an ounce of pain. Her hair reminds me of a warm safe place. Where as a child I'd hide. And pray for the thunder and the rain to quietly pass me by. Where do we go? Where do we go now? Where do we go? Ooh, oh, where do we go? Where do we go now? \U0001F3B8 https://www.youtube.com/watch?v=1w7OgIMMRc4"
texto_tweet = "My good friend @BarackObama knows what is at stake in the midterm elections. If we elect more Democrats to Congress, we can codify Roe v. Wade and move forwards, not backwards.\n\nGo to https://t.co/Hy8C4n0lUk to make your plan to vote. \nhttps://t.co/o8CQYr38fM"
print(texto)
print('\n',texto_tweet)

In [ ]:
# Primero seteamos algunos parametros que queremos que sean eliminados del string

# Stop words
stop_words = set(stopwords.words('english'))

# Signos de puntuación
signos_punt = string.punctuation

In [ ]:
def clean_text(text, is_tweet=False):
    '''
    Esta función limpia el caulquier texto. Elimina emoticones, emojis, palabras
    vacías (también llamadas stop words en la libreria de nltk), links, puntuaciones
    indicaciones de retweets, etc. Devuelve solo un string solo con las palabras  
    con mayor contenido para el analisis.
    Input:
        text (str): Texto original
        is_tweet (bool): si el texto es un tweet este parametro debe setearse a 
                         True para que también se limpie el texto con los 
                         caracteres más especificos de twitter (usa clean_tweet) 
    Output:
        text (str): Texto limpiado
    '''
    # definimos la acción si is_tweet == True - lo definimos primero sino algunos elementos que son especificos del tweets
    # (como por ejemplo el arrobado de personas) puede eliminarse mal 
    # (cuando borramos puntuaciones borramos @ y queda el nombre de usuario que luego no se borra al aplicar clean_tweets)
    if is_tweet:
        text = clean_tweet(text)

    # sacamos los emojis con codigo
    text = re.sub(emoji_pattern, '', text)
    # sacamos links
    text = re.sub(r'https\S+', '', text) # no utilizamos https.* porque eliminaba todo lo que seguía a un link, y puede pasar
                                         # que haya texto luego del link
    # sacamos signos de puntuación
    text = "".join([i.lower() for i in text if i not in signos_punt])
    # sacamos stop words
    tokens = text.split()
    text = " ".join([w for w in tokens if w not in stop_words])
 
    return text

In [ ]:
# Probamos que la función funciona correctamente

texto_limpio = clean_text(texto)
print("TEXTO ORIGINAL:\n", texto)
print("\nTEXTO LIMPIO:\n", texto_limpio)

tweet_limpio = clean_text(texto_tweet, is_tweet=True)
print("\nTWEET ORIGINAL:\n", texto_tweet)
print("\nTWEET LIMPIO:\n", tweet_limpio)

In [ ]:
#Comentario_J: la solución está muy bien. Algo bueno para afinar la limpieza de stopwords es pasar el texto a minusculas, porque
            # el set de stopwords también está en minuscula. Por ejemplo, "She" tendría que quedar afuera del texto limpio, pero
            # al tener mayúscula no se reconoció como stopword.
        
from nltk.corpus import stopwords       
stop_words_2 = set(stopwords.words('english'))
print("'She' es stopword?:", "She" in stop_words_2)
print("'she' es stopword?:", "She".lower() in stop_words_2)

#### Ejercicio 2:
Construyan una función que reciba una fecha a testear y dos fechas limite. La función debe chequear si la fecha a testear cae entre las dos fechas limites y devolver True cuando lo haga y False cuando no. 

Prueben si su función anda como esperaban con las fechas sugeridas a continuación (y con otras opciones de date para asegurarse que funcione bien):

In [ ]:
date = datetime.strptime("2020-11-03", '%Y-%m-%d')
start_time = datetime.strptime("2020-10-20", '%Y-%m-%d')
end_time = datetime.strptime("2020-11-18", '%Y-%m-%d')

In [ ]:
def filter_year_and_month(date, start_time, end_time):

    '''
    Recibe una fecha y devuelve True si la fecha de la fila está dentro de los días especificados como inicio y fin, caso contrario False.
    '''

    if date > start_time and date < end_time:
        return True
    else:
        return False

In [ ]:
# Probamos si la función funciona correctamente (con diferentes opciones de date)

date1 = datetime.strptime("2020-11-03", "%Y-%m-%d")
date2 = datetime.strptime("2020-10-19", "%Y-%m-%d")
date3 = datetime.strptime("2020-11-19", "%Y-%m-%d")
dates = [date1, date2, date3]

for date in dates:
    prueba = filter_year_and_month(date, start_time, end_time)
    print(date, ":", prueba)

#### Ejercicio 3:
Construyan una función que genere un indicador de sentimiento positivo o negativo.

Si estan trabajando con tweets en inglés pueden usar sentiment.polarity de TextBlob.

Si estan trabajando con tweets en Español pueden usar sentiment_analysis de sentiment_analysis_spanish

Prueben su función con el texto del ejercicio 1.

In [ ]:
def generate_sentiment(text, is_tweet=False):

    '''
    Esta función limpia el texto y analiza el sentimiento.
    Input:
        text (str): texto a limpiar y analizar.
    Output:
         sentimiento (float): cuanto menor sea el número, más negativo es el texto y, cuanto mayor, más positivo.
    '''

    # Limpiamos el texto
    text = clean_text(text, is_tweet)

    # Calculamos el sentimiento con el método TextBlob
    blob = TextBlob(text)
    sentiment = blob.sentiment
    polarity = sentiment.polarity

    return polarity

In [ ]:
# Probamos si la función funciona correctamente

polarity1 = generate_sentiment(texto)
print("El indicador de sentimiento del texto es:", polarity1)

polarity2 = generate_sentiment(texto_tweet, is_tweet=True)
print("El indicador de sentimiento del texto es:", polarity2)

#### Ejercicio 4:
Construyan una función que reciba un texto y detecte caritas creadas con caracteres y caritas de UNICODE. Debe devolver una tupla que contenga dos booleanos. El primer booleano indicará si había (True) o no (False) caritas tristes y el segundo si había (True) o no (False) caritas felices.

Prueben esta función con el texto del ejercicio 1.

In [ ]:
def detect_sad_happy_icons(text):
    '''
    Esta funcion detecte caritas creadas con caracteres y caritas de UNICODE
    Input:
        text (str): texto en el cual buscar las caritas
    Output:
        sad, happy (bool, bool): Indican si había (True) o no (False) caritas 
                                 tristes y felices
    '''

    tristes=False
    felices=False

    if re.search(emoji_pattern_sad,text):
        tristes=True
    if re.search(emoji_pattern_happy,text):
        felices=True
    for sad in emoticons_sad:
        sad=sad.replace('\\','\\\\')
        sad=sad.replace(')','\)')
        sad=sad.replace('(','\(')
        sad=sad.replace('*','\*')
        sad=sad.replace('[','\[')
        sad=sad.replace(']','\]')
        sad=sad.replace('|','\|')
        regex = re.compile(sad)
        if regex.search(text):
            tristes=True

    for hap in emoticons_happy:
        hap=hap.replace('\\','\\\\')
        hap=hap.replace(')','\)')
        hap=hap.replace('(','\(')
        hap=hap.replace('*','\*')
        hap=hap.replace('[','\[')
        hap=hap.replace(']','\]')
        regex = re.compile(hap)
        if regex.search(text):
            felices=True
 
    return (felices,tristes)

In [ ]:
# Probamos la función con textos alternativos

texto_alt_1 = "Emoticon happy :) \nEmoticon triste :( \nEmoji feliz \U0001F600 \nEmoji triste \U0001F62F \nMástextoasndlaasdlaksjhdlkasjdlaskdj"
texto_alt_2 = "Emoticon happy :D  Mástextoasndlaasdlaksjhdlkasjdlaskdj"

print('Texto 1 :', texto_alt_1)
print('Texto 2 :', texto_alt_2)
print('---------------')

print('Texto 1 :', detect_sad_happy_icons(texto_alt_1))
print('Texto 2 :', detect_sad_happy_icons(texto_alt_2))

In [ ]:
# Probamos la función con los textos del tp

print('Texto :', detect_sad_happy_icons(texto))
print('Texto Tweet:', detect_sad_happy_icons(texto_tweet))

#### Ejercicio 5:
Elijan la cuenta de twitter de algún usuario público y algún periodo o evento que pueda haber sido importante para esa persona/cuenta y le pueda haber hecho cambiar el sentimiento de sus post. Expliquen muy brevemente su elección.

Repitan los pasos que hemos realizado en clase para construir la tabla llamada `df_all_tweets` (la construimos en el ejercicio 1). 

No es necesario que copien las funciones vistas durante la clase sincrónica. Simplemente llamenlas a continuación con la nueva cuenta de twitter que quieran analizar. 

In [ ]:
# Creo dataframe con información de los tweets de Elon Musk desde el 01/09/2022
df_tweets_musk = save_tweets_text(get_all_tweets("elonmusk", datetime(2022, 9, 1, 0, 0, 0)), csv_file="tweets_musk.csv")

# Inspecciono el dataframe
df_tweets_musk

##### Expliquen muy brevemente su elección de cuenta y periodo acá:

Se selecciono como cuenta de twitter la de: 

### `Elon Musk`

Durante el último año, el empresario estuvo involucrado en rumores sobre la compra de una parte mayoritaria de las acciones de Twitter. En particular, a comienzos de octubre, estos rumores parecen haberse confirmado. Es por ello que nos parece interesante analizar si durante ese periodo el sentimiento de sus post se modificó

Se toma como incicio para el analisis de los tweets septiembre de este año, de manera de obtener un periodo previo a que la noticia tomo fuerza nuevamente durante octubre, y poder realizar un análisis pre y post evento

#### Ejercicio 6:
Construyan una función que utilice las 4 funciones creadas previamente (ej.1 al 4). Los argumentos de esta función deben ser un df y el rango de fechas que quieren analizar. 

Esta función debe limpiar el texto de los tweets, filtrar los tweets que correspondan a fechas dentro del rango que ustedes eligieron en el punto 5, calcular el sentimiento del texto y buscar si tiene caritas.

Finalmente, la función debe devolver un data frame que solo tenga tweets posteados en el periodo que ustedes eligieron, y que tenga 4 nuevas columnas, una para el texto limpio, otra para el sentimiento y dos más que indiquen si el texto tenía caritas tristes y felices. 

Utilicen esta función sobre el dataframe `df_all_tweets` que armaron en el punto 5. 

In [ ]:
def add_sentiment(df, start_time, end_time):
    '''
    Esta funcion debe limpia el texto de los tweets, filtra los tweets que
    correspondan al periodo entre start_time y end_time, calcula el sentimiento 
    del texto y buscar si tiene caritas.
    Input:
        df (dataframe): tabla con los tweets en una columa llamada text
        start_time (datetime): fecha del tweet más antiguo a conservar
        end_time (datetime): fecha del tweet más reciente a conservar
    Output:
        df (dataframe): df actualizado con el filtro y las nuevas columnas (estas
                        deben ser: clean_text, sentimiento, sad_face, happy_face).
    '''
    df_cleaned=df.copy()
    df_cleaned['clean_text']=''
    df_cleaned['happy_face']=False
    df_cleaned['sad_face']=False
    df_cleaned['drop']=False
    
    for i,tw in df_cleaned.iterrows():
        df_cleaned.at[i,'clean_text']=clean_text(tw['text'], True)
        
        #Remover filas con fecha no comprendida
        if filter_year_and_month(tw['created_at'], start_time, end_time)==False:
            df_cleaned.at[i,'drop']=True
        
        #Detectar emojis y emoticones en el text
        if detect_sad_happy_icons(tw['text'])[0]:
            df_cleaned.at[i,'happy_face']=True
            
        if detect_sad_happy_icons(tw['text'])[1]:
            df_cleaned.at[i,'sad_face']=True
            
        #Calcular polaridad
        df_cleaned.at[i,'sentimiento']=generate_sentiment(tw['text'],True)
        
    df_cleaned=df_cleaned[df_cleaned['drop']==False]
    df_cleaned.drop(['drop'],axis=1,inplace=True)
    df_cleaned=df_cleaned.reset_index(drop=True)
    return df_cleaned

In [ ]:
# utilizamos la función en el df creado anteriormente

#print(df_tweets_musk[220:240])
first_date = datetime(2022, 10, 1, 0, 0, 0)
last_date = datetime(2022, 10, 24, 0, 0, 0)
df_musk_cleaned=add_sentiment(df_tweets_musk, first_date,last_date)

df_musk_cleaned.head(20)

#### Ejercicio 7: 
Utilizado un criterio arbitratio creado por ustedes, agreguen una nueva columna al dataframe que se llame `indice_sentimiento` y combine las columnas `sentimiento`, `sad_face`, `happy_face` (con la ponderación que ustedes elijan).

Expliquen muy brevemente la elección de ponderación que hicieron.

In [ ]:
df_musk_cleaned['indice_sentimiento']=1*df_musk_cleaned.sentimiento + 0.2*df_musk_cleaned.happy_face.apply(int) - 0.2*df_musk_cleaned.sad_face.apply(int)
df_musk_cleaned.head(20)

##### Expliquen muy brevemente su elección:

Decidimos tomar el sentimiento original ponderado por 1 y, luego, con una ponderación de igual magnitud (20%) pero en sentido contrario, se incorpora la aparición de caritas felices y tristes. De esta manera, un tweet que contenga ambos emojis no modifica su sentimiento, pero un tweet que sólo contiene uno de los dos, se pondera más hacia esa dirección.

#### Ejercicio 8: 
Elijan un umbral (por ejemplo, si su `indice_sentimiento` va de 0 a 1 el umbral podía ser el 0.5). Agreguen al df una nueva columna llamada `positivo` que tome valores:
- 1 cuando el tweet tiene polarity >= umbral, 
- 0 cuando el tweet tiene polarity < umbral 

In [ ]:
# Primero chequeamos minimo y maximo de la columna indice_sentimiento

min_is      =   df_musk_cleaned['indice_sentimiento'].min()
max_is      =   df_musk_cleaned['indice_sentimiento'].max()

print('(min, MAX) :', '(', min_is, ',',max_is, ')')

In [ ]:
# utilizamos como umbral la media del minimo y maximo de la columna indice_sentimiento
umbral=(min_is+max_is)/2

# Creamos la columna positivo como True o False en función de si se cumple que el valor de indice_sentimiento es mayor o igual al umbral
df_musk_cleaned['positivo']=df_musk_cleaned['indice_sentimiento'] >= umbral
# Pasamos los valores a 0 y 1
df_musk_cleaned['positivo']=df_musk_cleaned['positivo'].apply(int)

df_musk_cleaned

#### Ejercicio 9: 
Realicen un gráfico de nube con las palabras de todos los textos limpios de los tweets que tienene `positivo` == 1

In [ ]:
# Generamos una lista con todas las palabras que aparecen en la columna text_clean del dataframe particionado en función de
# la columna "positivo" para luego realizar la nube
texto_musk_0 = ' '.join(df_musk_cleaned[df_musk_cleaned['positivo'] == 0]['clean_text'])
texto_musk_1 = ' '.join(df_musk_cleaned[df_musk_cleaned['positivo'] == 1]['clean_text'])

In [ ]:
# Generamos una imagen de nube de palabras para aquellos tweets que tienen la columna positivo igual a 1
wordcloud = WordCloud().generate(texto_musk_1)
plt.figure(figsize=(12,9))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

#### Ejercicio 10: 
Realicen un gráfico de nube con las palabras de todos los textos limpios de los tweets que tienene `positivo` == 0

In [ ]:
# Generamos una imagen de nube de palabras para aquellos tweets que tienen la columna positivo igual a 0
wordcloud = WordCloud().generate(texto_musk_0)
plt.figure(figsize=(12,9))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()